# Retrieve USGS WDFN data

*Reproducible Workflows with Open Water Data: NASA, NOAA, USGS River Data Access*, Session 5, Tuesday, November 17, 2026

**Learning goal:** Retrieve USGS WDFN data for a location of interest, describe what comes back using the course's shared concepts, and choose an access pattern that scales with your question.

USGS Water Data for the Nation (WDFN) publishes the observations USGS collects at thousands of monitoring locations across the country: river discharge and gage height, groundwater levels, water quality and more. You will find the stream monitoring locations around the Ohio River at Louisville, Kentucky, then retrieve the sensor record, daily values and field measurements from the gage on the Ohio, monitoring location `USGS-03294500`, for 15 March to 15 May 2025.

The slides explain the ideas, and this notebook is for running the code. Run the cells in order.

## Before you start

**Setup.** Follow the setup instructions in the HydroShare README, then choose the **Python (open-water-data)** kernel.

**Downloads and time.** The downloads are small: nothing is saved except the two figures, which are written to your working folder. The whole notebook takes about 1 minute.

**Results can change.** Provisional values can still be revised, and lists of monitoring locations and the ends of records grow. Your counts, end dates and recent values may differ from the ones quoted here: check `approval_status`.

## Choosing an access route

For reference while you work: the routes, and what each is for.

All the routes below use the same Water Data APIs, so the choice is mostly about *which function* matches your question. None of them needs a key, but USGS recommends one for repeated or large requests. If you do not have an API key your queries have a limit of 50 requests per IP address per hour. With an API key that limit increases to 1000 requests per IP address per hour.

| Use case | Recommended route | Key needed? | Scaling limits |
|---|---|---|---|
| Explore by clicking, spot-check a site | [WDFN website](https://waterdata.usgs.gov/) and [Water Data For The Nation - Explore](https://waterdata.usgs.gov/explore/) | No | Manual; not reproducible on its own |
| Find monitoring locations and what they measure | `waterdata.get_monitoring_locations`, `get_time_series_metadata`, `get_combined_metadata` | No (key recommended) | Results come in pages with a maximum of 50,000 rows per page; each page is one request |
| Daily values (e.g. daily mean discharge), any length of record | `waterdata.get_daily` | No (key recommended) | The whole record for a site in one query (paged for you) |
| The sensor record (every 5–15 minutes) | `waterdata.get_continuous` | No (key recommended) | Up to three years per query |
| Discharge measured by hydrographers in the field | `waterdata.get_field_measurements` | No (key recommended) | About one measurement a month per site |
| One day (or a short window) at many sites | `waterdata.get_daily` with no sites, a list of sites, or optionally a `bbox` | No (key recommended) | Fewer requests than iterating by site; mind the page/request count |
| Code that uses the legacy Water Services | `dataretrieval.nwis` | No | Being retired ([decommissioned February 22nd, 2027](https://waterdata.usgs.gov/blog/wdfn-waterservices-degradation/)); port it to `waterdata` |

## Credentials

**USGS Water Data API key (optional).** The USGS Water Data APIs work without a key, but a key raises your rate limit; over the limit, requests fail with HTTP `429 Too Many Requests` until the hour is up. Request one at the [USGS Water Data API signup page](https://api.waterdata.usgs.gov/signup/). `dataretrieval` reads it from the `API_USGS_PAT` environment variable and sends it for you. If your environment doesn't set that variable, uncomment the two lines below and paste your key when asked: it is kept only for this session and isn't saved in the notebook. Don't print the key or write it into a cell.

In [ ]:
import os

# Optional: set the key for this session only (it isn't saved in the notebook)
# from getpass import getpass
# os.environ["API_USGS_PAT"] = getpass("USGS API key: ")
print("API_USGS_PAT is set:", bool(os.getenv("API_USGS_PAT")))

**Live coding 1: find monitoring locations.** Run from here to the end of *What the Louisville gage records, and since when*, then go back to the slides.

## Define your query

Keep what you're asking for (place, site type, codes, dates) in variables at the top of your script, rather than typing them into the middle of each function call. Your query is then recorded in one spot and easy to change. For your own river, change these values.

In [ ]:
# what we're asking for
STATE, COUNTY = "Kentucky", "Jefferson County"
site = "USGS-03294500"                    # Ohio River at Louisville, KY (found below)
START, END = "2025-03-15", "2025-05-15"   # the April 2025 flood

## Programmatic data discovery

### Codes: parameters and statistics

`get_reference_table` returns each list of codes as a `pandas` DataFrame, so you can look them up in code rather than on a web page. Like every `waterdata` function, it returns a pair, `(DataFrame, metadata)`; `_` or `md` catches the metadata.

In [ ]:
from dataretrieval import waterdata

parameter_codes, _ = waterdata.get_reference_table("parameter-codes")
statistic_codes, _ = waterdata.get_reference_table("statistic-codes")

print(parameter_codes.loc[parameter_codes["parameter_code"].isin(["00060", "00065"]),
                          ["parameter_code", "parameter_name", "unit_of_measure"]])
print(statistic_codes.loc[statistic_codes["statistic_code"].isin(["00003", "00011"])])

You should see `00060` Discharge in `ft3/s` and `00065` Gage height in `ft`, then `00003` (MEAN) and `00011` (INSTANTANEOUS). `parameter_codes` has about 19,600 rows and `statistic_codes` about 3,200. Other reference tables work the same way, for example `"site-types"`, `"states"`, `"counties"`, `"hydrologic-unit-codes"` and `"agency-codes"`.

**Note**: the `unit_of_measure` for `Discharge` above says `ft3/s`, but when retrieving data, the unit is referred to as `ft^3/s`.

### Finding monitoring locations

#### Stream monitoring locations in a county

Which stream monitoring locations are in Jefferson County, Kentucky (Louisville)? The state and county come from the query cell above.

In [ ]:
site_info, md = waterdata.get_monitoring_locations(
    state_name=STATE,
    county_name=COUNTY,
    site_type="Stream",
)
print(site_info.shape)
print(site_info[["monitoring_location_id", "monitoring_location_name", "drainage_area"]].head())

This returns a `GeoDataFrame` with one row per monitoring location (120 on 2026-10-08) and 44 columns. Key columns are `monitoring_location_id` (the location identifier), `monitoring_location_name`, `site_type`, `drainage_area` (square miles) and `geometry` (a point you can map). The list includes inactive sites, and, as the first rows show, a few entries that aren't usable gages: on 2026-10-08, ten "STUDENT" creeks and five "TEST" entries (such as "SANDY TEST NO 1" and "LOUISVILLE TEST SITE"). This is one reason to look at discovery results before you download anything.

#### Which of them have daily mean discharge

`get_combined_metadata` can be used as an alternative to `get_monitoring_locations` to discover monitoring locations. It combines monitoring-location details with the list of time series each location records, so you can filter on parameter and statistic codes in the same query.

In [ ]:
sites_with_q, md = waterdata.get_combined_metadata(
    state_name=STATE,
    county_name=COUNTY,
    site_type="Stream",
    parameter_code="00060",   # discharge
    statistic_id="00003",     # daily mean
)
print(len(sites_with_q), "time series at", sites_with_q["monitoring_location_id"].nunique(), "locations")
print(sites_with_q[["monitoring_location_id", "monitoring_location_name", "begin", "end"]].head())

You should see about `29 time series at 27 locations`. Each row is one *time series*, not one location, so a location with two daily-discharge series appears twice. `begin` and `end` show the period each series covers: a series whose `end` is years ago belongs to a gage that no longer reports. One of these is `USGS-03294500`, Ohio River at Louisville, KY, with daily discharge from 1928 to the present.

#### Map: where these locations are

A quick map is a good check that discovery found what you meant. The river line on this map comes from the USGS NLDI web service, which returns the stream network up- and downstream of a monitoring location as GeoJSON. The cell also saves the map as `wdfn-ohio-louisville-map.png` in your working folder.

In [ ]:
import pandas as pd
import geopandas as gpd
import requests
import matplotlib.pyplot as plt

# site (Ohio River at Louisville) is set in the query block above

# leave out the 15 entries that are not real gages (names such as "STUDENT1 CREEK" or "SANDY TEST NO 1");
# this filter fits Jefferson County: look at your own list before reusing it
real = site_info[~site_info["monitoring_location_name"].str.contains("STUDENT|TEST")]
with_q = sites_with_q.drop_duplicates("monitoring_location_id")

# the Ohio River, 60 km up- and downstream of the gage (UM = upstream mainstem, DM = downstream mainstem)
parts = []
for nav in ["UM", "DM"]:
    url = f"https://api.water.usgs.gov/nldi/linked-data/nwissite/{site}/navigation/{nav}/flowlines"
    r = requests.get(url, params={"distance": 60}, timeout=120)
    r.raise_for_status()
    parts.append(gpd.GeoDataFrame.from_features(r.json()["features"], crs="EPSG:4326"))

river = pd.concat(parts)
fig, ax = plt.subplots(figsize=(7, 6))
river.plot(ax=ax, color="#2a78d6", linewidth=2)
real.plot(ax=ax, color="#b9b8b2", markersize=14, label=f"Stream monitoring locations ({len(real)})")
with_q.plot(ax=ax, color="#52514e", markersize=24, label=f"With daily mean discharge ({len(with_q)})")
gage = with_q[with_q["monitoring_location_id"] == site]
gage.plot(ax=ax, color="#eb6834", markersize=220, marker="*", edgecolor="white", linewidth=0.8, zorder=5,
          label=f"{site}, Ohio River at Louisville, KY")
# the label positions, limits and aspect below fit Jefferson County, KY: change or delete them for your area
ax.annotate("Ohio River", (-85.97, 38.03), fontsize=10, color="#2a78d6", style="italic")
ax.annotate("Louisville gage", gage.geometry.iloc[0].coords[0], xytext=(-100, 14), textcoords="offset points", fontsize=10)
ax.set_xlim(-86.05, -85.35)
ax.set_ylim(37.95, 38.42)
ax.set_aspect(1 / 0.786)   # roughly equal km in x and y at 38° N
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("USGS stream monitoring locations, Jefferson County, KY", fontsize=12, loc="left")
ax.legend(loc="lower right", fontsize=9)
fig.tight_layout()
fig.savefig("wdfn-ohio-louisville-map.png", dpi=150)

You should see the Ohio River as a blue line curving along the north and west edge of Jefferson County, about a hundred gray and dark gray dots east of the river (stream monitoring locations, and the darker ones with a daily mean discharge record), and an orange star for the Louisville gage on the river.

#### What the Louisville gage records, and since when

Once you've picked a location, ask which time series it records, and for what period. This is discovery for a single site:

In [ ]:
series, md = waterdata.get_time_series_metadata(monitoring_location_id=site)
print(series[["parameter_code", "parameter_name", "statistic_id", "computation_period_identifier", "begin", "end"]])

Each row is one time series. `statistic_id` `00011` with period `Points` is the continuous sensor record (gage height since 2007, discharge since 2009), and `00003` with period `Daily` is the daily mean (discharge since 1928). The Water Year rows are annual summaries (they start in 1832, long before the sensor record), the sediment series ran only from 1979 to 1983, and `00045` is a precipitation record added in 2026. Your `end` dates will be later.

Once you have your monitoring location ID(s) and codes, move on to downloads.

**Back to the slides.**

**Live coding 2: get WDFN data.** Run from here to the end of *Spatial scaling*, then go back to the slides. If your kernel restarted, use **Run All Above** from this cell first.

## Programmatic data downloads

Every `waterdata` function returns a `(DataFrame, metadata)` pair. The DataFrame holds the values; the metadata object records the request (`md.url`) and when it was made, which is useful when you write down how you got your data.

### Continuous data

`get_continuous` accepts up to three years per query. Continuous `time` values are UTC timestamps, so the request gives full timestamps ending in `Z` (UTC); daily values use the location's local calendar date, so the daily request below needs only plain dates. Here we request two months of discharge and gage height together:

In [ ]:
# START and END (the April 2025 flood) are set in the query block above
cont, md = waterdata.get_continuous(
    monitoring_location_id=site,
    parameter_code=["00060", "00065"],    # discharge and gage height
    time=f"{START}T00:00:00Z/{END}T23:59:59Z",
)
print(cont.shape)
print(cont[["time", "parameter_code", "value", "unit_of_measure", "approval_status", "qualifier"]].head())

The result is a long-format `GeoDataFrame`: one row per timestamp and parameter (nearly 6,000 rows for each of the two parameters, every 15 minutes), with columns including `monitoring_location_id`, `parameter_code`, `statistic_id`, `time` (UTC), `value`, `unit_of_measure`, `approval_status` and `qualifier`.

To find the flood crest, take the row with the largest value for each parameter:

In [ ]:
peaks = cont.loc[cont.groupby("parameter_code")["value"].idxmax()]
print(peaks[["parameter_code", "time", "value", "unit_of_measure", "approval_status"]])

The Ohio crested at **719,000 ft³/s** at 14:30 UTC on 9 April 2025 (10:30 a.m. in Louisville), with a gage height of **67.98 ft** half an hour earlier. Gage height is measured above the gage's own reference point (its datum), not from the riverbed, so it isn't the water depth. Times are in UTC: convert them before comparing with local records.

Look at `approval_status`, too. On 2026-10-08, values up to 9 April 2025 17:15 UTC were `Approved`, and everything after that was still provisional. Provisional data can still be revised when USGS reviews them ([USGS provisional data statement](https://waterdata.usgs.gov/provisional-data-statement/)), so check `approval_status` before you publish numbers, and expect values after the crest to change. No values in this window carry a `qualifier`.

### Daily data

Daily values are summaries of the continuous record, here the daily mean (`statistic_id="00003"`) discharge. The `time` argument can be a plain date range:

In [ ]:
daily, md = waterdata.get_daily(
    monitoring_location_id=site,
    parameter_code="00060",
    statistic_id="00003",     # daily mean
    time=f"{START}/{END}",
)
daily = daily.sort_values("time")
print(daily["approval_status"].value_counts().to_dict())
print(daily[["time", "value", "unit_of_measure", "approval_status", "qualifier"]].iloc[[0, 17, 25, 30, 61]])

There are 62 daily values, one per day (`{'Provisional': 37, 'Approved': 25}`). The daily mean peaked at 712,000 ft³/s on 9 April, slightly below the 719,000 ft³/s continuous peak, as expected for an average. That day is `Provisional` even though its morning values are approved, because the daily mean includes the provisional afternoon. Daily `time` is the monitoring location's local calendar date, not a timestamp.

### Field measurements

Field measurements are the discharge, gage-height, and groundwater level measurements that hydrographers make in person at the gage. USGS uses discharge and gage-height to build and check the rating curve that turns the sensor's gage height into the continuous discharge record, so they are the closest thing to "ground truth" for discharge:

In [ ]:
fm, md = waterdata.get_field_measurements(
    monitoring_location_id=site,
    time="2025-01-01T00:00:00Z/2025-12-31T00:00:00Z",
)
discharge_fm = fm[fm["parameter_code"] == "00060"].sort_values("time")
print(discharge_fm[["time", "value", "unit_of_measure", "observing_procedure", "measurement_rated", "approval_status"]])

Field measurements include both discharge (`00060`) and gage-height (`00065`) readings; we kept only discharge. In 2025, hydrographers measured discharge five times, including on the day of the crest. `observing_procedure` records how each measurement was made, here always with an acoustic Doppler current profiler (ADCP) towed across the river. `measurement_rated` is a qualitative estimate of the quality of a measurement: the hydrographer's judgment of its accuracy. The crest-day measurement (738,000 ft³/s) is rated `Poor`, which is common for measurements in fast, debris-laden flood water. It is about 3% above the continuous record's peak, well within what a `Poor` rating allows. The field measurement `time` is a date; the time of day is in a separate `time_of_day` column.

### Plot the data

Plotting the continuous record by approval status, with the field measurement on top, shows the flood, the crest and where provisional data begin. The cell also saves the plot as `wdfn-ohio-louisville-discharge.png`.

In [ ]:
import matplotlib.dates as mdates

fig, ax = plt.subplots(figsize=(8, 4.2))
q = cont[cont["parameter_code"] == "00060"].sort_values("time")
colors = {"Approved": "#2a78d6", "Provisional": "#eb6834"}
for status, part in q.groupby("approval_status"):
    ax.plot(part["time"], part["value"] / 1000, color=colors[status], linewidth=2, label=f"Continuous discharge ({status})")

in_window = discharge_fm[(discharge_fm["time"] >= START) & (discharge_fm["time"] <= END)]
ax.scatter(in_window["time"], in_window["value"] / 1000, s=60, color="#0b0b0b", zorder=5, label="Field measurement")
crest = q.loc[q["value"].idxmax()]
ax.annotate(f"Crest {crest['value']:,.0f} ft³/s\n{crest['time']:%d %b %H:%M} UTC", (crest["time"], crest["value"] / 1000),
            xytext=(18, -8), textcoords="offset points", fontsize=9)
ax.set_ylabel("Discharge (thousand ft³/s)")
ax.set_ylim(0, 800)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
ax.set_title(f"Ohio River at Louisville, KY ({site}), spring 2025", fontsize=12, loc="left")
ax.legend(frameon=False, fontsize=9, loc="center right", bbox_to_anchor=(1.0, 0.62))
fig.tight_layout()
fig.savefig("wdfn-ohio-louisville-discharge.png", dpi=150)

You should see discharge around 100,000 to 200,000 ft³/s in March, rising steeply in early April to a crest of 719,000 on 9 April, then falling back to about 100,000 by late April. The line is blue (approved) up to the crest and orange (provisional) after it, and a black dot marks the crest-day field measurement of 738,000 (plotted at its date; it is rated `Poor`).

## Understanding what you downloaded

The continuous values, daily values and field measurements share their core columns. Here is how they map onto the course's shared concepts, using what came back for Louisville:

| Shared concept | What it is in your download | Column(s) |
|---|---|---|
| Location identifier | `USGS-03294500`: agency prefix plus site number | `monitoring_location_id` |
| Variable | discharge (`00060`) and gage height (`00065`) | `parameter_code`, plus `statistic_id` (`00011` continuous, `00003` daily mean) |
| Variable unit | `ft^3/s` for discharge, `ft` for gage height | `unit_of_measure` |
| Time | continuous: UTC timestamps every 5–15 minutes (15 at Louisville); daily: a calendar date; field measurements: a date plus `time_of_day` | `time` |
| Data quality flag(s) | approval status (`Approved`, or `Provisional` and subject to revision); `qualifier` for special conditions (e.g. `[ESTIMATED]`, `[ICE]`); `measurement_rated` (`Good`, `Fair`, `Poor`) for field measurements | `approval_status`, `qualifier`, `measurement_rated` |
| Version / provenance | No version number: the service you called, the request URL and the date you retrieved the data. Values can change until they're approved; `last_modified` says when each was last changed | `md.url`, your retrieval date, `last_modified` |
| Data unit | one time series per monitoring location, parameter and statistic | `time_series_id` |

**Provisional data and re-running later.** Because provisional values can change, the same request run next year may return different numbers for April 2025. Record the date you retrieved the data, and save what you downloaded.

**Keep raw and derived data apart.** Save each download unchanged (for example `data/raw/usgs_03294500_continuous_2025-03-15_2025-05-15.csv`), and write anything you compute from it, such as the crest table or a cleaned series, to a separate folder (`data/derived/`). Then you can always trace a number back to the download it came from (Data Management, Module 1).

**Cite what you used.** USGS water data have a DOI, [https://doi.org/10.5066/F7P55KJN](https://doi.org/10.5066/F7P55KJN) (U.S. Geological Survey, National Water Information System). Cite it with the monitoring location IDs, parameter codes, dates and your access date, and cite `dataretrieval` too: Hodson and Hariharan, [https://doi.org/10.5066/P94I5TX3](https://doi.org/10.5066/P94I5TX3). Data Publishing in Module 1 covers citing data.

## Best practices FAQs

### Temporal scaling

**What is the recommended way to access data for one location but the full period of record?**

Make one request per site and leave out `time`. For daily values, `get_daily` then returns the whole record, and `dataretrieval` handles the paging. The Louisville gage has daily discharge back to 1928:

In [ ]:
daily_record, md = waterdata.get_daily(
    monitoring_location_id=site,
    parameter_code="00060",
    statistic_id="00003",
)
print(len(daily_record), daily_record["time"].min(), daily_record["time"].max())
print(daily_record.loc[daily_record["value"].idxmax(), ["time", "value"]])

On 2026-10-08 this one request returned 36,054 rows, one per day from 1 January 1928 to the day before, in about 6 seconds. The largest daily mean in the record is 1,110,000 ft³/s on 27 January 1937. Look at the `qualifier` column on long records: about 1,100 of these days are marked `[ESTIMATED]` ("Going further" at the end of this notebook counts them), some together with `[ICE]` or `[EQUIP]` (equipment problems).

Continuous values are limited to three years per query, so request a long continuous record in three-year windows (the Louisville record starts in 2009, so that's six chunks of calls).

### Spatial scaling

**What is the recommended way to download data across all locations but a small time range?**

Leave out `monitoring_location_id` and set a short `time` instead. One request for one day returns that day's value for every site with daily mean discharge. Looping over thousands of site IDs one at a time would quickly use up your rate limit.

In [ ]:
one_day, md = waterdata.get_daily(
    parameter_code="00060",
    statistic_id="00003",
    time="2025-04-09",    # one date; also accepts an interval such as "2025-04-01/2025-04-10"
)
print(len(one_day), "rows at", one_day["monitoring_location_id"].nunique(), "locations")

On 2026-10-08 this returned 9,041 daily values at 9,029 locations (a few locations have more than one discharge series) in a couple of seconds. To narrow the area, add a `bbox` (west, south, east, north, in decimal degrees) or a list of sites rather than looping:

In [ ]:
around_louisville, md = waterdata.get_daily(
    parameter_code="00060",
    statistic_id="00003",
    time="2025-04-09",
    bbox=[-86.5, 37.8, -85.0, 38.8],
)
print(around_louisville.sort_values("value", ascending=False)[["monitoring_location_id", "value", "approval_status"]].head(3))

That returned 36 locations in the box. The two largest are the Louisville gage (712,000 ft³/s) and a second Ohio River location in Louisville, at the water tower (`USGS-03292494`, 668,000).

**Back to the slides.**

## Now you try it

Repeat the main steps for accessing USGS WDFN data on a second river: the **Willamette River at Salem, Oregon**.

**Your task:** Retrieve discharge from a monitoring location on the Willamette using `dataretrieval`.

Make your changes in the code cells below. They use new names (ending in `_salem`), so your Louisville results stay as they are and you can compare them. Fill in each `...`, using the code above. If a cell stops with an error, there's probably still a `...` to fill in.

### Step 1: the location and its time series

`site_salem` is set for you (`USGS-14191000`): get its time series metadata to check the expected record lengths.

In [ ]:
site_salem = "USGS-14191000"                  # Willamette River at Salem, OR

series_salem, md = ...                        # TODO: get the time series metadata for site_salem
print(series_salem[["parameter_code", "parameter_name", "statistic_id", "computation_period_identifier", "begin", "end"]])

<details>
<summary><b>Show the example answer</b></summary>

```python
site_salem = "USGS-14191000"                  # Willamette River at Salem, OR

series_salem, md = waterdata.get_time_series_metadata(monitoring_location_id=site_salem)
print(series_salem[["parameter_code", "parameter_name", "statistic_id", "computation_period_identifier", "begin", "end"]])
```

</details>

**What you should get:** discharge records going back to 1986 for continuous values (`00060`, `00011`, `Points`), and 1909 for daily values (`00060`, `00003`, `Daily`). The list also has other parameters, such as water temperature (`00010`) and specific conductance (`00095`); you only need discharge (`00060`) and gage height (`00065`).

### Step 2: daily mean discharge

The dates `START_salem, END_salem` are set for you (`"2026-02-01", "2026-03-31"`). Get the daily mean discharge for 1 February to 31 March 2026. Find the highest daily mean, and check its approval status.

In [ ]:
START_salem, END_salem = "2026-02-01", "2026-03-31"

daily_salem, md = ...                         # TODO: daily mean discharge (00060, 00003) at site_salem, START_salem to END_salem
print(len(daily_salem), daily_salem["approval_status"].value_counts().to_dict())
print(daily_salem.loc[daily_salem["value"].idxmax(), ["time", "value", "unit_of_measure", "approval_status"]])

<details>
<summary><b>Show the example answer</b></summary>

```python
START_salem, END_salem = "2026-02-01", "2026-03-31"

daily_salem, md = waterdata.get_daily(monitoring_location_id=site_salem, parameter_code="00060",
                                      statistic_id="00003", time=f"{START_salem}/{END_salem}")
print(len(daily_salem), daily_salem["approval_status"].value_counts().to_dict())
print(daily_salem.loc[daily_salem["value"].idxmax(), ["time", "value", "unit_of_measure", "approval_status"]])
```

</details>

**What you should get:** 59 daily values, almost all `Provisional`; the highest daily mean is **62,200 ft³/s** on 25 February 2026.

### Step 3: the continuous record and the crest

Get the continuous discharge and gage height for the same window and find the crest, as in the continuous-values example.

In [ ]:
cont_salem, md = ...                          # TODO: continuous discharge and gage height at site_salem for the same window (UTC times)
peaks_salem = ...                             # TODO: the row with the largest value for each parameter
print(peaks_salem[["parameter_code", "time", "value", "unit_of_measure", "approval_status"]])

<details>
<summary><b>Show the example answer</b></summary>

```python
cont_salem, md = waterdata.get_continuous(monitoring_location_id=site_salem, parameter_code=["00060", "00065"],
                                          time=f"{START_salem}T00:00:00Z/{END_salem}T23:59:59Z")
peaks_salem = cont_salem.loc[cont_salem.groupby("parameter_code")["value"].idxmax()]
print(peaks_salem[["parameter_code", "time", "value", "unit_of_measure", "approval_status"]])
```

</details>

**What you should get:** the continuous record crests on 25 February at about **64,000 ft³/s**, with gage height near **16.9 ft**. The two peaks needn't share a timestamp: on 2026-11-11, discharge peaked at 63,900 ft³/s at 22:55 UTC and gage height at 16.85 ft at 23:45 UTC. Salem's record here comes every 5 minutes, Louisville's every 15: the interval differs by site. Because these values are provisional, your numbers may differ slightly.

**Back to the slides.**

## Going further

### Counting qualifiers on long records

*Optional.* This goes beyond the session. It runs on its own (it doesn't use earlier results). "Run All" stops at the unfinished task above, so run these cells one at a time (Shift+Enter).

This downloads the whole daily record for the Ohio River at Louisville again, as in the "Temporal scaling" example (about 6 seconds), and counts its qualifiers:

In [ ]:
from dataretrieval import waterdata

# the whole daily record, as in "Temporal scaling" (the site is written out so this cell runs on its own)
daily_record, md = waterdata.get_daily(
    monitoring_location_id="USGS-03294500",
    parameter_code="00060",
    statistic_id="00003",
)
print(daily_record["qualifier"].astype(str).str.contains("ESTIMATED").sum(), "days marked [ESTIMATED]")
daily_record["qualifier"].astype(str).value_counts()

**What you should see:** about 1,100 days marked `[ESTIMATED]`, then one row per combination of qualifiers, most often `['ESTIMATED']` alone. Each day's qualifiers are a list, shown here as text, and days with no qualifier are left out of the counts.

## Further reading

- `dataretrieval` (Python) GitHub repo: [github.com/DOI-USGS/dataretrieval-python](https://github.com/DOI-USGS/dataretrieval-python)
- [`dataretrieval` documentation](https://doi-usgs.github.io/dataretrieval-python/)
- [USGS Water Data APIs documentation](https://api.waterdata.usgs.gov/)
- [USGS Water Data API keys](https://api.waterdata.usgs.gov/docs/ogcapi/keys/)
- [USGS Water Data for the Nation](https://waterdata.usgs.gov/)
- [Water Data For The Nation - Explore](https://waterdata.usgs.gov/explore/) (GUI)
- [USGS provisional data statement](https://waterdata.usgs.gov/provisional-data-statement/)
- [NWIS WaterServices decommissioning](https://waterdata.usgs.gov/blog/wdfn-waterservices-degradation/)

## About this notebook

Released under the [GNU General Public License v3.0](https://www.gnu.org/licenses/gpl-3.0.html) (GPL-3.0). You may use, change and share it under the same license.

Adapted from [Notebook to Demonstrate Collecting USGS Data](https://github.com/CUAHSI/notebooks/tree/develop/Data%20Access%20Examples/USGS%20-%20Plotting%20Streamflow%20using%20NWIS%20DataRetrieval) by CUAHSI, CUAHSI notebooks (GPL-3.0). Its legacy `nwis` calls are ported to `waterdata` here.

**How to cite the course:** Platt, L., Wong, K., Vaarre-Lamoureux, K., & Bogan, A. *Reproducible Workflows with Open Water Data: NASA, NOAA, USGS River Data Access* (Version pilot-2026-11) [Learning material (online course)]. CUAHSI. GPL-3.0.

**How to cite the data:** USGS water data have a DOI, [https://doi.org/10.5066/F7P55KJN](https://doi.org/10.5066/F7P55KJN) (U.S. Geological Survey, National Water Information System). Cite it with the monitoring location IDs, parameter codes, dates and your access date, and cite `dataretrieval` too: Hodson and Hariharan, [https://doi.org/10.5066/P94I5TX3](https://doi.org/10.5066/P94I5TX3).

**Acknowledgements:** this course was developed by CUAHSI with contributions from colleagues at NOAA, USGS and NASA. This material is based upon work supported by the U.S. National Science Foundation under [Award No. 2535162](https://www.nsf.gov/awardsearch/show-award?AWD_ID=2535162). Any opinions, findings, and conclusions or recommendations expressed in this material are those of the authors and do not necessarily reflect the views of the National Science Foundation. Generative AI (Anthropic's Claude) was used to help draft content, and contributors and staff reviewed, edited and tested all of it.